In [ ]:
# Imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import pdb # for debugging

import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/renatomaaliw3/public_files/refs/heads/master/Data%20Sets/Apriori%20Data.csv')
df

,Technical Skills,Professional Certifications,Communication Skills,Grit,Work Habits,Employability
0,Technical Skills - Fair,Professional Certifications - 1,Communication Skills - 3,Grit - 1,Work Habits - 3,Less Employable
1,Technical Skills - Fair,Professional Certifications - 1,Communication Skills - 4,Grit - 2,Work Habits - 4,Employable
2,Technical Skills - Very Good,Professional Certifications - 2,Communication Skills - 5,Grit - 2,Work Habits - 4,Employable
3,Technical Skills - Fair,Professional Certifications - 2,Communication Skills - 4,Grit - 2,Work Habits - 3,Employable
4,Technical Skills - Good,Professional Certifications - 1,Communication Skills - 4,Grit - 2,Work Habits - 4,Employable
...,...,...,...,...,...,...
495,Technical Skills - Fair,Professional Certifications - 2,Communication Skills - 5,Grit - 2,Work Habits - 5,Employable
496,Technical Skills - Fair,Professional Certifications - 1,Communication Skills - 5,Grit - 1,Work Habits - 3,Underemployed
497,Technical Skills - Excellent,Professional Certifications - 2,Communication Skills - 4,Grit - 2,Work Habits - 3,Employable
498,Technical Skills - Good,Professional Certifications - 1,Communication Skills - 5,Grit - 1,Work Habits - 4,Underemployed


In [ ]:
# Data Preprocessing
# Before Applying the Apriori algorithm, we need to preprocess the data
# One-Hot Encoding, Remember get dummies?

from mlxtend.preprocessing import TransactionEncoder

# Consolidate each transaction into a single list of items, removing NaN values
transactions = df.apply(lambda row: row.dropna().tolist(), axis = 1).tolist()

# Initialize TransactionEncoder
encoder = TransactionEncoder()

# Fit and transform the transactions data
transaction_matrix = encoder.fit_transform(transactions)

# Convert to DataFrame
transaction_df = pd.DataFrame(transaction_matrix, columns = encoder.columns_)
transaction_df

,Communication Skills - 2,Communication Skills - 3,Communication Skills - 4,Communication Skills - 5,Employable,Grit - 1,Grit - 2,Less Employable,Professional Certifications - 1,Professional Certifications - 2,Technical Skills - Excellent,Technical Skills - Fair,Technical Skills - Good,Technical Skills - Pass,Technical Skills - Very Good,Underemployed,Work Habits - 3,Work Habits - 4,Work Habits - 5
0,False,True,False,False,False,True,False,True,True,False,False,True,False,False,False,False,True,False,False
1,False,False,True,False,True,False,True,False,True,False,False,True,False,False,False,False,False,True,False
2,False,False,False,True,True,False,True,False,False,True,False,False,False,False,True,False,False,True,False
3,False,False,True,False,True,False,True,False,False,True,False,True,False,False,False,False,True,False,False
4,False,False,True,False,True,False,True,False,True,False,False,False,True,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,False,False,False,True,True,False,True,False,False,True,False,True,False,False,False,False,False,False,True
496,False,False,False,True,False,True,False,False,True,False,False,True,False,False,False,True,True,False,False
497,False,False,True,False,True,False,True,False,False,True,True,False,False,False,False,False,True,False,False
498,False,False,False,True,False,True,False,False,True,False,False,False,True,False,False,True,False,True,False


In [ ]:
# Appying the Apriori Algorithm
# Since data are cleaned and prepared for frequent itemset

from mlxtend.frequent_patterns import apriori, association_rules

# Apply the Apriori algorithm
frequent_itemsets = apriori(transaction_df, min_support = 0.2, use_colnames = True)

# min_support is the minimum support threshold. Itemsets with support greater than or equal to this threshold will be returned.
# use_colnames = True ensures that the item names are used in the output instead of column indices.

In [ ]:
# Generate Association Rules

rules = association_rules(frequent_itemsets, num_itemsets = len(transaction_df), metric = "confidence", min_threshold = 0.2)

rules.loc[:, :'lift']
# rules.loc[:, :'lift'].to_csv('rules.csv')

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
0,(Employable),(Communication Skills - 4),0.546,0.486,0.268,0.490842,1.009964
1,(Communication Skills - 4),(Employable),0.486,0.546,0.268,0.551440,1.009964
2,(Grit - 1),(Communication Skills - 4),0.478,0.486,0.232,0.485356,0.998674
3,(Communication Skills - 4),(Grit - 1),0.486,0.478,0.232,0.477366,0.998674
4,(Grit - 2),(Communication Skills - 4),0.522,0.486,0.254,0.486590,1.001214
...,...,...,...,...,...,...,...
199,"(Underemployed, Professional Certifications - 1)","(Technical Skills - Fair, Grit - 1)",0.294,0.396,0.238,0.809524,2.044252
200,(Technical Skills - Fair),"(Grit - 1, Professional Certifications - 1, Un...",0.542,0.294,0.238,0.439114,1.493586
201,(Grit - 1),"(Technical Skills - Fair, Underemployed, Profe...",0.478,0.238,0.238,0.497908,2.092050
202,(Professional Certifications - 1),"(Technical Skills - Fair, Grit - 1, Underemplo...",0.520,0.238,0.238,0.457692,1.923077


## **Answers:**

# **Q17: What itemset/s has the least support?**

In [ ]:
least_support_itemset = frequent_itemsets.sort_values(by = 'support').iloc[0]
least_support_itemset

,46
support,0.202
itemsets,"(Grit - 2, Work Habits - 4, Employable)"


# ***Itemset: (Grit - 2, Work Habits - 4, Employable)***

# **Q18: According to Association Rules, what is the most superior rule that makes an applicant 'Employable'?**

In [ ]:
# Filter rules where the consequent is 'Employable'
employable_rules = rules[rules['consequents'].apply(lambda x: 'Employable' in x)]

# Find the rule with the highest confidence
most_superior_rule = employable_rules.sort_values(by = 'confidence', ascending = False).iloc[0]

print("The most superior rule that makes an applicant 'Employable' based on confidence is:")
print(f"Rule: {list(most_superior_rule['antecedents'])} -> {list(most_superior_rule['consequents'])}")
print(f"Confidence: {most_superior_rule['confidence']:.4f}")
print(f"Lift: {most_superior_rule['lift']:.4f}")

The most superior rule that makes an applicant 'Employable' based on confidence is:
Rule: ['Professional Certifications - 2', 'Communication Skills - 4'] -> ['Employable']
Confidence: 1.0000
Lift: 1.8315


# ***Most superior rule: {'Professional Certifications - 2', 'Communication Skills - 4'} -> {'Employable'}***

# **Q19: Does {'Underemployed'} and {'Grit-1'} have strong assocation? Explain**

In [ ]:
underemployed_grit_rules = rules[
    (rules['antecedents'].apply(lambda x: list(x) == ['Underemployed']) &
     rules['consequents'].apply(lambda x: list(x) == ['Grit - 1'])) |
    (rules['antecedents'].apply(lambda x: list(x) == ['Grit - 1']) &
     rules['consequents'].apply(lambda x: list(x) == ['Underemployed']))
]

if not underemployed_grit_rules.empty:
    print("Association rules between {'Underemployed'} and {'Grit - 1'} (as sole items):")
    display(underemployed_grit_rules.loc[:, :'lift'])

Association rules between {'Underemployed'} and {'Grit - 1'} (as sole items):


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
30,(Grit - 1),(Underemployed),0.478,0.294,0.294,0.615063,2.09205
31,(Underemployed),(Grit - 1),0.294,0.478,0.294,1.000000,2.09205


# ***There is a positive association between these itemsets as the lift value is greater than 1.***

# **Q20: Does {'Grit - 2', 'Technical Skills - Good'} makes you {'Employable'}? Explain**

In [ ]:
# Find rules with {'Grit - 2', 'Technical Skills - Good'} as antecedent and {'Employable'} as consequent
grit2_techgood_employable_rule = rules[
    rules['antecedents'].apply(lambda x: set(x) == {'Grit - 2', 'Technical Skills - Good'}) &
    rules['consequents'].apply(lambda x: set(x) == {'Employable'})
]

if not grit2_techgood_employable_rule.empty:
    print("Association rule for {'Grit - 2', 'Technical Skills - Good'} -> {'Employable'}:")
    display(grit2_techgood_employable_rule.loc[:, :'lift'])

    # Explain the association based on metrics
    rule = grit2_techgood_employable_rule.iloc[0]
    confidence = rule['confidence']
    lift = rule['lift']

    print(f"\nRule: {list(rule['antecedents'])} -> {list(rule['consequents'])}")
    print(f"Confidence: {confidence:.4f}")
    print(f"Lift: {lift:.4f}")

Association rule for {'Grit - 2', 'Technical Skills - Good'} -> {'Employable'}:


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
100,"(Grit - 2, Technical Skills - Good)",(Employable),0.292,0.546,0.292,1.0,1.831502



Rule: ['Grit - 2', 'Technical Skills - Good'] -> ['Employable']
Confidence: 1.0000
Lift: 1.8315


# ***There is a positive association between {'Grit - 2', 'Technical Skills - Good'} and {'Employable'}. As the lift value is greater than 1, it is positive that an applicant with 'Grit - 2' and 'Technical Skills - Good' makes them 'Employable'.***